# Topic 79 — Model C: TCN Neural Network (GPU)

**BTC/USD 15-Minute Volatility Prediction using a Temporal Convolutional Network**

Run this on Google Colab with a T4 GPU runtime for fast training.

After training, download `predict_79_model_c.pkl` and deploy with:
```bash
TOPIC_ID=79 PREDICT_PKL=predict_79_model_c.pkl python deploy_worker_raw.py
```

---
**Runtime setup:** Runtime → Change runtime type → T4 GPU

In [ ]:
# Install dependencies
!pip install -q git+https://github.com/allora-network/allora-forge-builder-kit.git@feature/topic-79-volatility-target
!pip install -q cloudpickle torch numpy pandas scikit-learn

In [ ]:
import torch
print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# === CONFIGURATION ===
# Paste your Allora API key here (get free at https://developer.allora.network)
ALLORA_API_KEY = "UP-xxx"  # <-- PASTE YOUR KEY

TICKERS = ["btcusd"]
DAYS_OF_HISTORY = 800  # 2+ years — GPU makes this feasible
INTERVAL = "1m"
NUMBER_OF_INPUT_BARS = 30
TARGET_BARS = 15
TARGET_TYPE = "volatility"

# Training config (GPU-optimized)
EPOCHS = 80
BATCH_SIZE = 8192
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
PATIENCE = 12

In [ ]:
import os
os.environ["ALLORA_API_KEY"] = ALLORA_API_KEY

import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from sklearn.model_selection import TimeSeriesSplit
import torch
import torch.nn as nn
import cloudpickle
from allora_forge_builder_kit import AlloraMLWorkflow, PerformanceEvaluator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## Model Architecture

Temporal Convolutional Network with dilated causal convolutions.
Receptive field covers the full 30-bar input with dilations [1, 2, 4, 8, 16].

In [ ]:
class CausalConv1d(nn.Module):
    """Causal convolution: output at time t only depends on inputs at t and before."""
    def __init__(self, in_channels, out_channels, kernel_size, dilation=1):
        super().__init__()
        self.padding = (kernel_size - 1) * dilation
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size,
                              padding=self.padding, dilation=dilation)

    def forward(self, x):
        out = self.conv(x)
        if self.padding > 0:
            out = out[:, :, :-self.padding]
        return out


class TCNBlock(nn.Module):
    """Residual TCN block with dilated causal convolution."""
    def __init__(self, channels, kernel_size, dilation, dropout=0.05):
        super().__init__()
        self.net = nn.Sequential(
            CausalConv1d(channels, channels, kernel_size, dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
            CausalConv1d(channels, channels, kernel_size, dilation),
            nn.BatchNorm1d(channels),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return x + self.net(x)


class VolatilityTCN(nn.Module):
    """
    TCN for volatility prediction.
    Input: (batch, seq_len, 5) — normalized OHLCV bars
    Output: (batch, 1) — predicted volatility
    """
    def __init__(self, input_features=5, seq_len=30, hidden_channels=128,
                 kernel_size=3, dilations=(1, 2, 4, 8, 16), dropout=0.05):
        super().__init__()
        self.input_proj = nn.Conv1d(input_features, hidden_channels, 1)
        self.tcn_blocks = nn.ModuleList([
            TCNBlock(hidden_channels, kernel_size, d, dropout) for d in dilations
        ])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(hidden_channels, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
            nn.Softplus(),  # non-negative output
        )

    def forward(self, x):
        x = x.transpose(1, 2)  # (B, seq, feat) -> (B, feat, seq)
        x = self.input_proj(x)
        for block in self.tcn_blocks:
            x = block(x)
        return self.head(x)


# Count parameters
model_test = VolatilityTCN()
n_params = sum(p.numel() for p in model_test.parameters())
print(f"Model parameters: {n_params:,}")
del model_test

## Data Loading & Backfill

In [ ]:
print("[1/4] Initializing workflow...")
workflow = AlloraMLWorkflow(
    tickers=TICKERS,
    number_of_input_bars=NUMBER_OF_INPUT_BARS,
    target_bars=TARGET_BARS,
    interval=INTERVAL,
    target_type=TARGET_TYPE,
    data_source="allora",
    api_key=ALLORA_API_KEY,
)

print(f"[2/4] Backfilling {DAYS_OF_HISTORY} days...")
start_date = datetime.now(timezone.utc) - timedelta(days=DAYS_OF_HISTORY)
workflow.backfill(start=start_date)
print("✅ Backfill complete")

In [ ]:
print("[3/4] Preparing tensors...")
df_all = workflow.get_full_feature_target_dataframe(start_date=start_date).reset_index()
base_feature_cols = [col for col in df_all.columns if col.startswith("feature_")]
df_all = df_all.dropna(subset=base_feature_cols + ["target"])

n_samples = len(df_all)
seq_len = NUMBER_OF_INPUT_BARS
n_features = 5

# Build (samples, seq_len, 5) array
X_seq = np.zeros((n_samples, seq_len, n_features), dtype=np.float32)
for i in range(seq_len):
    X_seq[:, i, 0] = df_all[f"feature_open_{i}"].values
    X_seq[:, i, 1] = df_all[f"feature_high_{i}"].values
    X_seq[:, i, 2] = df_all[f"feature_low_{i}"].values
    X_seq[:, i, 3] = df_all[f"feature_close_{i}"].values
    X_seq[:, i, 4] = df_all[f"feature_volume_{i}"].values

y_all = df_all["target"].values.astype(np.float32)

# Pre-allocate GPU tensors
X_tensor = torch.from_numpy(X_seq).to(device)
y_tensor = torch.from_numpy(y_all).unsqueeze(1).to(device)

print(f"✅ Dataset: {n_samples:,} samples on {device}")
print(f"   Tensor shape: {X_tensor.shape}")
print(f"   GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB" if torch.cuda.is_available() else "")

## Training with Walk-Forward CV

In [ ]:
print("[4/4] Training...")

VAL_BATCH = 16384  # chunk size for validation (avoids OOM)

tscv = TimeSeriesSplit(n_splits=3, gap=TARGET_BARS)
fold_predictions = np.full(n_samples, np.nan)

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X_seq)):
    print(f"\n{'='*60}")
    print(f"Fold {fold_idx+1}/3: Train={len(train_idx):,}, Test={len(test_idx):,}")
    print(f"{'='*60}")

    # Slice tensors (already on GPU)
    X_train = X_tensor[train_idx]
    y_train = y_tensor[train_idx]
    X_test = X_tensor[test_idx]
    y_test = y_tensor[test_idx]

    n_train = len(train_idx)
    n_batches = (n_train + BATCH_SIZE - 1) // BATCH_SIZE

    model = VolatilityTCN(
        input_features=n_features,
        seq_len=seq_len,
        hidden_channels=128,
        kernel_size=3,
        dilations=(1, 2, 4, 8, 16),
        dropout=0.05,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

    best_val_loss = float("inf")
    patience_counter = 0
    best_state = None

    for epoch in range(EPOCHS):
        model.train()
        perm = torch.randperm(n_train, device=device)
        train_loss = 0.0
        for bi in range(n_batches):
            idx = perm[bi * BATCH_SIZE : (bi + 1) * BATCH_SIZE]
            xb = X_train[idx]
            yb = y_train[idx]
            pred = model(xb)
            loss = nn.MSELoss()(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item() * len(idx)
        train_loss /= n_train
        scheduler.step()

        # Validate (chunked to avoid OOM)
        model.eval()
        with torch.no_grad():
            val_preds = []
            for vi in range(0, len(test_idx), VAL_BATCH):
                val_preds.append(model(X_test[vi:vi+VAL_BATCH]))
            val_pred = torch.cat(val_preds)
            val_loss = nn.MSELoss()(val_pred, y_test).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_counter = 0
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        else:
            patience_counter += 1

        if (epoch + 1) % 5 == 0:
            lr_now = optimizer.param_groups[0]['lr']
            print(f"  Epoch {epoch+1:3d}: train={train_loss:.8f} val={val_loss:.8f} best={best_val_loss:.8f} lr={lr_now:.6f}")

        if patience_counter >= PATIENCE:
            print(f"  Early stopping at epoch {epoch+1} (best val_loss={best_val_loss:.8f})")
            break

    # Predict on test set (chunked)
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        chunks = []
        for vi in range(0, len(test_idx), VAL_BATCH):
            chunks.append(model(X_test[vi:vi+VAL_BATCH]).cpu())
        test_preds = torch.cat(chunks).numpy().flatten()
    fold_predictions[test_idx] = test_preds
    print(f"  ✅ Fold {fold_idx+1} done. Best val_loss: {best_val_loss:.8f}")

In [ ]:
# Evaluate CV results
valid_mask = ~np.isnan(fold_predictions)
evaluator = PerformanceEvaluator()
metrics = evaluator.evaluate(y_true=y_all[valid_mask], y_pred=fold_predictions[valid_mask])

print(f"\n{'='*60}")
print(f"CV RESULTS: {metrics['num_passed']}/7 ({metrics['score']:.1%} — {metrics['grade']})")
print(f"{'='*60}")
evaluator.print_report(metrics, detailed=False)

## Train Final Model on All Data

In [ ]:
print("Training final model on all data...")

final_model = VolatilityTCN(
    input_features=n_features,
    seq_len=seq_len,
    hidden_channels=128,
    kernel_size=3,
    dilations=(1, 2, 4, 8, 16),
    dropout=0.05,
).to(device)

n_batches_all = (n_samples + BATCH_SIZE - 1) // BATCH_SIZE
optimizer = torch.optim.AdamW(final_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=20, T_mult=2)

for epoch in range(EPOCHS):
    final_model.train()
    perm = torch.randperm(n_samples, device=device)
    epoch_loss = 0.0
    for bi in range(n_batches_all):
        idx = perm[bi * BATCH_SIZE : (bi + 1) * BATCH_SIZE]
        xb = X_tensor[idx]
        yb = y_tensor[idx]
        pred = final_model(xb)
        loss = nn.MSELoss()(pred, yb)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(final_model.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item() * len(idx)
    epoch_loss /= n_samples
    scheduler.step()
    if (epoch + 1) % 10 == 0:
        print(f"  Epoch {epoch+1:3d}: loss={epoch_loss:.8f}")

final_model.eval()
final_model_cpu = final_model.cpu()
print(f"\n✅ Final model trained on {n_samples:,} samples")

## Create Predict Function & Save Pickle

In [ ]:
# The predict function runs on CPU at inference time (worker deployment)
def predict(nonce=None):
    """Predict 15-min BTC volatility using the TCN model."""
    live_row = workflow.get_live_features(ticker=TICKERS[0])
    if live_row is None or len(live_row) == 0:
        raise ValueError("Could not get live features")

    x = np.zeros((1, NUMBER_OF_INPUT_BARS, 5), dtype=np.float32)
    row = live_row.iloc[0]
    for i in range(NUMBER_OF_INPUT_BARS):
        x[0, i, 0] = row[f"feature_open_{i}"]
        x[0, i, 1] = row[f"feature_high_{i}"]
        x[0, i, 2] = row[f"feature_low_{i}"]
        x[0, i, 3] = row[f"feature_close_{i}"]
        x[0, i, 4] = row[f"feature_volume_{i}"]

    x_tensor = torch.tensor(x, dtype=torch.float32)
    with torch.no_grad():
        vol = final_model_cpu(x_tensor).item()

    vol = max(0.0, vol)
    print(f"Model C (TCN) prediction: {vol:.6f} (15-min vol)")
    return vol


# Test it
print("Testing prediction...")
test_pred = predict()
print(f"\n✅ Test prediction: {test_pred:.6f}")

In [ ]:
# Save the pickle
with open("predict_79_model_c.pkl", "wb") as f:
    cloudpickle.dump(predict, f)

import os
pkl_size = os.path.getsize("predict_79_model_c.pkl") / 1024
print(f"✅ Saved predict_79_model_c.pkl ({pkl_size:.0f} KB)")
print(f"   CV Score: {metrics['num_passed']}/7 ({metrics['score']:.1%} — {metrics['grade']})")
print(f"   Architecture: TCN (128ch, dilations 1/2/4/8/16)")
print(f"   Training samples: {n_samples:,}")

In [ ]:
# Download the pickle file
try:
    from google.colab import files
    files.download("predict_79_model_c.pkl")
    print("📥 Download started!")
except ImportError:
    print("Not running on Colab — file saved locally as predict_79_model_c.pkl")